### **Intall Dependencies**

In [ ]:
"""
Author: Aquiles Elbaum

Description:
    Batch inference and attention extraction pipeline using Gemma-3 for
    landmine sensor recommendation, designed for execution in Google Colab.
"""

!pip install -q transformers accelerate datasets torch pandas

# **Confirm GPU and RAM**
You can change the Google Collab runtime:


*   -> Runtime
*   -> Change runtime type
* -> T4 GPU if on free version, A100 if pro




In [ ]:
# Confirm GPU and RAM
!nvidia-smi
!torch.cuda.get_device_name(0)

# **Load data from google drive folder**

In [ ]:
from google.colab import drive, files
import pandas as pd
drive.mount('/content/drive')
df = pd.read_csv("/content/drive/MyDrive/Capstone/Data/mines.csv")
df

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,Context,Target
0,"Voltage: 0.338156758 V, Height: 0 cm, Soil Typ...",Mine Type: NA
1,"Voltage: 0.320241334 V, Height: 0.181818182 cm...",Mine Type: NA
2,"Voltage: 0.28700875 V, Height: 0.272727273 cm,...",Mine Type: NA
3,"Voltage: 0.256283622 V, Height: 0.454545455 cm...",Mine Type: NA
4,"Voltage: 0.262839599 V, Height: 0.545454545 cm...",Mine Type: NA
...,...,...
333,"Voltage: 0.323262478 V, Height: 0.909090909 cm...",Mine Type: M14 Anti-Personnel
334,"Voltage: 0.444108237 V, Height: 0.181818182 cm...",Mine Type: M14 Anti-Personnel
335,"Voltage: 0.353473918 V, Height: 0.454545455 cm...",Mine Type: M14 Anti-Personnel
336,"Voltage: 0.36253735 V, Height: 0.727272727 cm,...",Mine Type: M14 Anti-Personnel


# **Select precision based on GPU**

In [ ]:
import os
import torch
import pandas as pd
from datetime import datetime
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from datasets import Dataset
import json

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    major, minor = torch.cuda.get_device_capability()
    print(f"GPU detected: {gpu_name} (Compute capability {major}.{minor})")

    # A100 / L4 GPUs support bf16
    if major >= 8:
        torch_dtype = torch.bfloat16
    else:
        torch_dtype = torch.float16
else:
    print("No GPU detected, falling back to CPU.")
    torch_dtype = torch.float32

# Cache model
os.environ["TRANSFORMERS_CACHE"] = "/content/cache"

GPU detected: NVIDIA A100-SXM4-80GB (Compute capability 8.0)


# **Run Model**
Outputs are saved to google drive in csv format

In [ ]:
from huggingface_hub import login

login("YOUR TOKEN")

# Load model and tokenizer
model_id = "google/gemma-3-4b-it"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch_dtype,
    device_map="auto"
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
    batch_size=8,             # Increase based on GPU capacity
    truncation=True,
    padding=True,
    temperature=0.7,
    max_new_tokens=100,
    return_full_text=False
)


# Load data
df = df.head(80)
dataset = Dataset.from_pandas(df)

# Build all the prompts at once
def build_prompt(batch):
    return {
        "prompt": [
            f"You are a sensor expert. Given the following situation:\n"
            f"Context: {c}\nTarget: Land mine {t}\n\n"
            f"Recommend the most appropriate type of sensor to detect the target and briefly explain why. Be concise."
            for c, t in zip(batch["Context"], batch["Target"])
        ]
    }

dataset = dataset.map(build_prompt, batched=True, num_proc=4)

# Run inference
@torch.inference_mode()
def generate_recommendations(batch):
    outputs = pipe(batch["prompt"])
    return {"sensor_recommendation": [o[0]["generated_text"] for o in outputs]}

dataset = dataset.map(generate_recommendations, batched=True, batch_size=8)

# Save output
df_out = dataset.to_pandas()
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
output_path = f"/content/drive/MyDrive/Capstone/Output/sensor_recommendations_{timestamp}.csv"

os.makedirs(os.path.dirname(output_path), exist_ok=True)
df_out.to_csv(output_path, index=False)

print(f"\nSaved to: {output_path}")
print(f"Preview:")
print(df_out.head(3))

# **Saving Attention Weights**
The transformers.pipeline() is designed for high level generation, not for low level model internals like attentions or logits. So we can't use them here.



In [ ]:
import os
import torch
import pandas as pd
from datetime import datetime
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from datasets import Dataset
import json
import zipfile
from tqdm import tqdm

model_id = "google/gemma-3-4b-it"
torch_dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch_dtype,
    device_map="auto"
)
model.eval()
model.set_attn_implementation("eager")

# Load data
df = pd.read_csv("/content/drive/MyDrive/Capstone/Data/mines.csv")
# Uncomment to test on smaller dataset
# df = df.head(8)
dataset = Dataset.from_pandas(df)

# Build all the prompts at once
def build_prompt(batch):
    return {
        "prompt": [
            f"You are a sensor expert. Given the following situation:\n"
            f"Context: {c}\nTarget: Land mine {t}\n\n"
            f"Recommend the most appropriate type of sensor to detect the target and briefly explain why. Be concise."
            for c, t in zip(batch["Context"], batch["Target"])
        ]
    }

dataset = dataset.map(
    build_prompt,
    batched=True,
    num_proc=4
)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
base_dir = f"/content/drive/MyDrive/Capstone/Output/{timestamp}"
attn_dir = os.path.join(base_dir, "attentions")
os.makedirs(attn_dir, exist_ok=True)

# Run inference and save attention weights
@torch.inference_mode()
def generate_with_attentions(batch, indices):
    responses, attn_files = [], []

    for prompt, idx in zip(batch["prompt"], indices):
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, padding=True).to(model.device)

        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.7,
            return_dict_in_generate=True,
            output_attentions=True
        )

        full_response = tokenizer.decode(outputs.sequences[0], skip_special_tokens=True).strip()
        responses.append(full_response)

        # Save attention weights from final generation step
        if outputs.attentions is not None:
            final_step = outputs.attentions[-1]  # last token generation
            mean_layers = []
            for layer_attn in final_step:
                if isinstance(layer_attn, torch.Tensor):
                    mean_layers.append(layer_attn.mean(dim=(0, 1)))  # mean over batch & heads

            mean_layers = [layer.cpu().tolist() for layer in mean_layers]
            attn_path = os.path.join(attn_dir, f"attn_{idx:04d}.json")
            with open(attn_path, "w") as f:
                json.dump({"mean_attention_final_step": mean_layers}, f)
            attn_files.append(attn_path)
        else:
            attn_files.append(None)

    return {"full_response": responses, "attention_file": attn_files}

# Run inference
dataset = dataset.map(
    generate_with_attentions,
    with_indices=True,
    batched=True,
    batch_size=16
)

# Save responses and attention matrices
df_out = dataset.to_pandas()[["full_response", "attention_file"]]
csv_path = os.path.join(base_dir, "sensor_recommendations.csv")
df_out.to_csv(csv_path, index=False)

# Zip attention Jsons
zip_path = os.path.join(base_dir, "attentions_mean.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    for file in tqdm(os.listdir(attn_dir), total=len(os.listdir(attn_dir)), leave=False):
        full_path = os.path.join(attn_dir, file)
        zipf.write(full_path, arcname=file)

print(f"CSV: {csv_path}")
print(f"Mean attentions: {attn_dir}")
print(f"Zip archive: {zip_path}")
print(df_out.head(3))
